# LangChain Chat Models and Tools: Reference Notebook

## Introduction
This notebook is a guided reference for common LangChain patterns: sending a prompt to a chat model, streaming a response, supplying role-based messages, exposing a Python function as a tool, completing a tool-call round trip, and requesting structured output with Pydantic.

Run the cells from top to bottom. The examples make network requests and require the appropriate provider credentials. The stock-price tool uses hard-coded sample data; it does not retrieve live market prices. Keep API keys in environment variables or a local `.env` file, and do not commit credentials.

In [1]:
# Import the model, message, tool, and validation types used in later examples.
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from pydantic import BaseModel, Field

# Load local environment variables; override=True lets .env values replace existing ones.
load_dotenv(override=True)


True

## 1. Basic chat calls and streaming
The next examples create a chat-model client, send a complete prompt, and then stream a response incrementally.

In [2]:
# Create a reusable chat-model client for the examples that follow.
llm = ChatOpenAI(model='gpt-5.4-mini')

message = 'In 1 sentence, what does it mean for an AI Agent to be autonomous?'

# invoke() waits for the full response and returns an AIMessage.
reply = llm.invoke(message)

print(reply.content)

An AI agent is autonomous if it can independently perceive its environment, make decisions, and take actions to achieve goals with minimal human intervention.


In [3]:
# stream() yields response chunks as they arrive instead of waiting for completion.
for chunk in llm.stream("Tell me a two line poem about autonomous agents"):
    # Print chunks without adding a newline between them, and flush for immediate display.
    print(chunk.content, 
            end="",
            flush=True)

Autonomous agents roam where data streams ignite,  
Thinking in loops of purpose, they learn to choose their flight.

In [4]:
# Route ChatOpenAI through OpenRouter and select a provider-specific model.
openrouter_llm = ChatOpenAI(
    model='anthropic/claude-haiku-4.5',
    base_url='https://openrouter.ai/api/v1',
    api_key=os.getenv('OPENROUTER_API_KEY')
)

# This request uses OPENROUTER_API_KEY from the environment.
reply = openrouter_llm.invoke('In one sentence, what is Langchain?')
print(reply.content)

LangChain is a Python framework that simplifies building applications with large language models by providing tools for chaining together LLM calls, managing prompts, and integrating external data sources.


### Summary
- `invoke()` returns a complete message; `stream()` yields incremental chunks.
- A provider can be configured through the model name, API base URL, and an environment-based API key.

In [5]:
# Represent a conversation as messages with explicit system and user roles.
messages = [
    SystemMessage("You are a terse assistant who answers in exactly five words."),
    HumanMessage("What is the capital of France?")
]

print(llm.invoke(messages).content)

# The same roles and content can also be supplied as dictionaries.
messages_as_dicts = [
    {
        'role': 'system',
        'content': 'You are a terse assistant who answers in exactly five words'
    },
    {
        'role': 'user',
        'content': 'What is the capital of France?'
    }
]

print(llm.invoke(messages_as_dicts).content)

Paris is France’s capital city.
Paris is France’s capital city.


### Summary
Typed message objects and role/content dictionaries both express a conversation. The system message guides the response, while the human message contains the request.

## 2. Defining and calling tools
A tool gives the model a declared function it may request. The application still runs the function and sends its result back to the model.

In [6]:
# @tool exposes the function and its type/docstring as a schema for LangChain.
@tool
def get_share_price(symbol: str)->float:
    """Return a hard-coded example share price for the given symbol."""
    # These are fixed teaching examples, not live quotes.
    fake_prices = {
        'AAPL': 241.5,
        'GOOG': 168.2,
        'AMZN':198.0
    }
    # Normalize symbol case; unsupported symbols currently use 0.0 as a fallback.
    return fake_prices.get(symbol.upper(), 0.0)

print('name', get_share_price.name)
print('description', get_share_price.description)
print('args:', get_share_price.args)
print('called directly', get_share_price.invoke({'symbol': 'AAPL'}))

name get_share_price
description Return the current share price for the given symbol
args: {'symbol': {'title': 'Symbol', 'type': 'string'}}
called directly 241.5


In [7]:
# Bind the tool schema so the model can request this function by name.
llm_with_tools = llm.bind_tools([get_share_price])

# The model may return a tool call instead of a user-facing text response.
response = llm_with_tools.invoke('What is the share price of Amazon?')
print('content:', repr(response.content))
print('tool_calls:', response.tool_calls)

content: ''
tool_calls: [{'name': 'get_share_price', 'args': {'symbol': 'AMZN'}, 'id': 'call_malubKmXipAPBoGjHM00KD6r', 'type': 'tool_call'}]


In [8]:
# Start the conversation and keep the model's tool request in the history.
conversation = [HumanMessage('What is the share price of Amazon?')]
ai_message = llm_with_tools.invoke(conversation)
conversation.append(ai_message)

# The application executes each recognized request and correlates its result by call ID.
for call in ai_message.tool_calls:
    if call['name'] == 'get_share_price':
        result = get_share_price.invoke(call['args'])
        conversation.append(ToolMessage(content=str(result), 
                            tool_call_id=call['id']))
# Invoke again, now that the model can see the tool result.
final = llm_with_tools.invoke(conversation)
print(final.content)

Amazon (AMZN) is **198.0**.


### Summary and logic notes
The tool workflow has three steps: bind the tool schema, let the model request a call, then execute the call in application code and return its result as a `ToolMessage` linked by the call ID. A final model invocation turns the tool result into a user-facing answer.

This example dispatches only `get_share_price`. The sample function also returns `0.0` for an unsupported ticker, which can look like a real quote; treat that behavior as a teaching simplification, not production error handling.

## 3. Structured output
Use a Pydantic model when downstream code needs named fields with predictable types instead of free-form prose.

In [9]:
# Define the fields and types expected in the model's structured response.
class Company(BaseModel):
    name: str = Field(description='The company name')
    ticker: str = Field(description='The stock ticker symbol')
    founded_year: int = Field(description='The year the company was founded')

# Configure the model to parse its response into a validated Company instance.
structured_llm = llm.with_structured_output(Company)

company = structured_llm.invoke('Tell me about Amazon the technology company')
print(company)
print("Just the ticker:", company.ticker)

name='Amazon' ticker='AMZN' founded_year=1994
Just the ticker: AMZN


### Summary
`with_structured_output(Company)` asks LangChain to parse the model response into the declared schema. The returned object exposes validated fields such as `company.ticker` for later code.

## Conclusion
These examples progress from direct and streamed chat calls to structured messages, tool execution, and schema-based responses. The key distinction is that a model can request a tool, but application code is responsible for executing it, validating or handling its result, and sending that result back. For real applications, replace sample data with a trusted source and handle unsupported tools, missing data, and provider errors explicitly.